## MScFE 610 Financial Econometrics — Group Work Project 1
#### Group 17817 · Module 6 · Technical notebook

This notebook holds the code and output for every computational answer in the report. Each section
is headed by the question it answers. Group 17817 is an **odd** group, so Problem 3 uses
`FE-GWP1_model_selection_1.csv`. Figures are written to `figures/`, data are read from `data/`.

| Section | Question | Content |
|---|---|---|
| Setup | — | Libraries, master seed, plotting conventions |
| 1 | **1e** | Omitted-variable bias by Monte Carlo simulation |
| 2 | **2b, 2c** | Sensitivity of OLS to outliers; the high-leverage limit |
| 3 | **3a, 3b** | Model selection on dataset 1; consistency of BIC versus AIC |
| 4 | **4c–4h** | Interpretation of the log-wage equation |
| 5 | **5c** | Unit roots, Johansen cointegration tests, VECM |
| 6 | **6a, 6b** | AR(1) MA(∞) representation; AR(2) autocorrelations |

Questions that are purely analytical (1a–1d, 2a, 4a–4b, 5a–5b) are answered in the written report.

#### Setup

In [ ]:
import itertools, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM, select_coint_rank, select_order
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox, linear_reset
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130, "display.float_format", lambda v: f"{v:9.4f}")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.autolayout": True})
BLUE, RED, GREEN, PURPLE, GREY = "#2A6FDB", "#D1495B", "#3C8D5A", "#8C6BB1", "#6B7280"

SEED = 17817          # the group number, used as the master seed so every result is reproducible
DATA = "data"         # input CSV files
FIG  = "figures"      # every chart is saved here for the report
os.makedirs(FIG, exist_ok=True)

def save_fig(name):
    """Save the current figure to figures/<name>.png at print resolution, then display it."""
    plt.savefig(f"{FIG}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

print("Master random seed:", SEED)

### Question 1e — Omitted-variable bias in a controlled simulation

Data-generating process, with every parameter known by construction:

$$Y_i = a + bX_i + cZ_i + e_i,\qquad a=1,\; b=2,\; c=1.5,$$

where $(X_i, Z_i)$ are drawn from a bivariate normal with unit variances and correlation
$\rho=0.7$, and $e_i \sim N(0,1)$ is drawn independently of both regressors. We then estimate the
model twice: once correctly, and once with $Z$ omitted.

Theory predicts that omitting $Z$ drives the estimator of $b$ to
$\; \operatorname{plim}\hat b_{\text{short}} = b + c\,\dfrac{\operatorname{Cov}(X,Z)}{\operatorname{Var}(X)} = 2 + 1.5(0.7) = 3.05.$

In [ ]:
# Known data-generating process: Y = a + bX + cZ + e, with X and Z correlated by construction
A_TRUE, B_TRUE, C_TRUE = 1.0, 2.0, 1.5        # true intercept and slopes
RHO, SX, SZ, SE = 0.7, 1.0, 1.0, 1.0          # corr(X, Z), sd(X), sd(Z), sd(e)

def draw(n, rng, rho=RHO):
    """One sample of size n. e is drawn independently of X and Z, as the question requires."""
    cov = [[SX**2, rho*SX*SZ], [rho*SX*SZ, SZ**2]]
    XZ = rng.multivariate_normal([0, 0], cov, size=n)
    X, Z = XZ[:, 0], XZ[:, 1]
    Y = A_TRUE + B_TRUE*X + C_TRUE*Z + rng.normal(0, SE, n)
    return X, Z, Y

# Omitting Z shifts the slope on X by c * Cov(X,Z)/Var(X) = c * rho * sd(Z)/sd(X)
theory_bias = C_TRUE * RHO * SZ / SX
print(f"True b = {B_TRUE};  theoretical asymptotic value of the misspecified estimator "
      f"= b + c*rho = {B_TRUE + theory_bias:.4f}")

In [ ]:
# --- 1e(i) one sample of 200 observations, estimated both ways -------------------------------
rng = np.random.default_rng(SEED)
X, Z, Y = draw(200, rng)

# Fit the correct model and the misspecified one on the same data
full  = sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit()
short = sm.OLS(Y, sm.add_constant(X)).fit()
aux   = sm.OLS(Z, sm.add_constant(X)).fit()          # auxiliary regression of the omitted variable on X

print("Correct specification   Y ~ X + Z")
print(f"   a = {full.params[0]:.4f} (se {full.bse[0]:.4f})   b = {full.params[1]:.4f} (se {full.bse[1]:.4f})"
      f"   c = {full.params[2]:.4f} (se {full.bse[2]:.4f})   R2 = {full.rsquared:.4f}   s = {np.sqrt(full.mse_resid):.4f}")
print("Misspecified            Y ~ X          (Z omitted)")
print(f"   a = {short.params[0]:.4f} (se {short.bse[0]:.4f})   b = {short.params[1]:.4f} (se {short.bse[1]:.4f})"
      f"                        R2 = {short.rsquared:.4f}   s = {np.sqrt(short.mse_resid):.4f}")
print(f"\nAuxiliary regression Z on X: slope = {aux.params[1]:.4f}")

# The omitted-variable formula holds exactly in-sample, not only on average
print(f"Algebraic identity  b_short = b_full + c_full * slope(Z on X):"
      f"  {short.params[1]:.6f}  =  {full.params[1] + full.params[2]*aux.params[1]:.6f}")
print(f"\nCorrelation of the short-model residual with X = {np.corrcoef(short.resid, X)[0,1]:.2e}  (zero by construction)")
print(f"Correlation of the short-model residual with Z = {np.corrcoef(short.resid, Z)[0,1]:.4f}  "
      "(the omitted variable now lives inside the disturbance)")

In [ ]:
# --- 1e(ii) does a larger sample fix the problem?  5,000 replications at each sample size -----
NS, REP = [25, 50, 100, 200, 500, 1000, 5000, 20000], 5000
rows = []
for n in NS:
    rng = np.random.default_rng(SEED + n)                  # separate, reproducible stream per n
    bf, bs, sef, ses = [], [], [], []                      # slope and its s.e.: full / short model
    for _ in range(REP):
        X, Z, Y = draw(n, rng)
        f = sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit()    # correct model
        s = sm.OLS(Y, sm.add_constant(X)).fit()                          # Z omitted
        bf.append(f.params[1]); bs.append(s.params[1]); sef.append(f.bse[1]); ses.append(s.bse[1])
    bf, bs, sef, ses = map(np.array, (bf, bs, sef, ses))
    # bias = mean error; RMSE combines bias and spread; cover95 = share of 95% intervals containing b
    rows.append(dict(n=n,
                     mean_correct=bf.mean(), sd_correct=bf.std(ddof=1), bias_correct=bf.mean()-B_TRUE,
                     rmse_correct=np.sqrt(((bf-B_TRUE)**2).mean()), cover95_correct=np.mean(np.abs(bf-B_TRUE) <= 1.96*sef),
                     mean_omitted=bs.mean(), sd_omitted=bs.std(ddof=1), bias_omitted=bs.mean()-B_TRUE,
                     rmse_omitted=np.sqrt(((bs-B_TRUE)**2).mean()), cover95_omitted=np.mean(np.abs(bs-B_TRUE) <= 1.96*ses)))
mc = pd.DataFrame(rows)
print(mc.to_string(index=False))
print(f"\nAsymptotic target for the misspecified estimator: {B_TRUE + theory_bias:.4f}")

In [ ]:
# --- 1e(iii) the bias tracks corr(X, Z) exactly ------------------------------------------------
# For each correlation, fit the short model (Z omitted) on 2,000 large samples and average the slope.
sens = []
for r in [-0.9, -0.5, -0.2, 0.0, 0.2, 0.5, 0.7, 0.9]:
    rng = np.random.default_rng(SEED + int(1000*r) + 7)
    bs = []
    for _ in range(2000):
        X, Z, Y = draw(5000, rng, rho=r)
        bs.append(sm.OLS(Y, sm.add_constant(X)).fit().params[1])
    sens.append(dict(corr_XZ=r, simulated_mean_b=np.mean(bs), predicted_b=B_TRUE + C_TRUE*r,
                     simulated_bias=np.mean(bs)-B_TRUE, predicted_bias=C_TRUE*r))
sens = pd.DataFrame(sens)
print(sens.to_string(index=False))

In [ ]:
# --- 1e figure ---------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))

# 3,000 samples at n = 200 give the sampling distribution of each estimator
rng = np.random.default_rng(SEED + 1)
bf_s, bs_s = [], []
for _ in range(3000):
    X, Z, Y = draw(200, rng)
    bf_s.append(sm.OLS(Y, sm.add_constant(np.column_stack([X, Z]))).fit().params[1])
    bs_s.append(sm.OLS(Y, sm.add_constant(X)).fit().params[1])

# (i) histogram of the two estimators against the true b and its biased limit
ax[0].hist(bf_s, 45, alpha=.75, color=BLUE, label="Both predictors included")
ax[0].hist(bs_s, 45, alpha=.75, color=RED,  label="Z omitted")
ax[0].axvline(B_TRUE, color="k", lw=1.6, label=f"True b = {B_TRUE}")
ax[0].axvline(B_TRUE+theory_bias, color=RED, ls="--", lw=1.6, label=f"b + c·ρ = {B_TRUE+theory_bias:.2f}")
ax[0].set(xlabel="Estimate of b", ylabel="Frequency out of 3,000 samples",
          title="(i) Sampling distributions at n = 200")
ax[0].legend(fontsize=7, frameon=False)

# (ii) mean +/- 1.96 SD across sample sizes, taken from the Monte Carlo table
ax[1].axhline(B_TRUE, color="k", lw=1.5, label=f"True b = {B_TRUE}")
ax[1].axhline(B_TRUE+theory_bias, color=RED, ls="--", lw=1.5, label=f"Asymptotic limit = {B_TRUE+theory_bias:.2f}")
ax[1].errorbar(mc.n, mc.mean_correct, yerr=1.96*mc.sd_correct, fmt="o-", color=BLUE, ms=4, capsize=2,
               label="Both predictors included")
ax[1].errorbar(mc.n, mc.mean_omitted, yerr=1.96*mc.sd_omitted, fmt="s-", color=RED, ms=4, capsize=2,
               label="Z omitted")
ax[1].set(xscale="log", xlabel="Sample size n (logarithmic scale)", ylabel="Mean estimate of b ± 1.96 SD",
          title="(ii) A larger sample does not remove the bias")
ax[1].legend(fontsize=7, frameon=False)

# (iii) simulated against predicted limit for each corr(X, Z)
ax[2].plot(sens.corr_XZ, sens.simulated_mean_b, "o", color=RED, ms=6, label="Simulated mean, Z omitted")
ax[2].plot(sens.corr_XZ, sens.predicted_b, "-", color="k", lw=1.4, label="Theory: b + c·corr(X, Z)")
ax[2].axhline(B_TRUE, color=GREY, ls=":", lw=1.4, label=f"True b = {B_TRUE}")
ax[2].set(xlabel="corr(X, Z)", ylabel="Mean estimate of b",
          title="(iii) Sign and size of the bias follow corr(X, Z)")
ax[2].legend(fontsize=7, frameon=False)
save_fig("fig_1e_omitted_variable_bias")

#### Baseline replication — single draw per sample size (Answer.pdf)

The group's first pass at 1e used a different known model, $Y_i = 2 + 1.5X_i + 2Z_i + e_i$, with
$X_i = 0.7Z_i + \sqrt{1-0.7^2}\,u_i$ so that $\operatorname{corr}(X,Z)=0.7$, and one sample at each
size. Theory gives $\operatorname{plim}\hat b_{\text{short}} = 1.5 + 2(0.7) = 2.9$. The cell reproduces
the table in the written answer exactly (legacy seed 610).

In [ ]:
np.random.seed(610)          # legacy global seed, kept so the numbers match Answer.pdf exactly
rows = []
for n in [100, 500, 1000, 5000]:
    Zb = np.random.normal(0, 1, n)
    Xb = 0.7*Zb + np.sqrt(1 - 0.7**2)*np.random.normal(0, 1, n)   # unit variance, corr(X, Z) = 0.7
    Yb = 2 + 1.5*Xb + 2*Zb + np.random.normal(0, 1, n)            # e independent of X and Z
    # slope on X with Z included, then with Z omitted
    rows.append(dict(n=n,
                     b_with_X_and_Z=sm.OLS(Yb, sm.add_constant(np.column_stack([Xb, Zb]))).fit().params[1],
                     b_with_X_only=sm.OLS(Yb, sm.add_constant(Xb)).fit().params[1]))
print("True b = 1.5;  probability limit with Z omitted = 1.5 + 2(0.7) = 2.9\n")
print(pd.DataFrame(rows).to_string(index=False))

**Results.** With Z included, the estimate of $b$ is centred on the true value 2 at every sample size (bias below 0.004) and its spread shrinks like $1/\sqrt{n}$. With Z omitted, it is centred on $b + c\rho = 3.05$ at every sample size: the bias of about 1.05 is the same at $n = 25$ and at $n = 20{,}000$. Only the spread shrinks, so the 95% interval covers the true $b$ in 9% of samples at $n = 25$ and in none from $n = 200$ onwards. **A larger sample does not cure omitted-variable bias; it makes the wrong answer more precise.** The bias moves one-for-one with $c\cdot\operatorname{corr}(X,Z)$ and disappears only when $\operatorname{corr}(X,Z) = 0$. The baseline replication shows the same pattern with a different model: $\hat b \approx 2.8$–$2.9$ against a true 1.5, with no improvement as $n$ grows.

### Question 2b — How outliers move the fitted line

Construction of the simulation. Sixty clean observations are generated from
$y_i = 2 + 1.5x_i + \varepsilon_i$ with $x_i \sim U(0,10)$ and $\varepsilon_i \sim N(0,1)$. That clean
sample is then held **fixed** and contaminated in four different ways, so that every difference in the
estimates is attributable to the contamination alone and not to resampling noise:

| Scenario | Contamination | Design position |
|---|---|---|
| S0 | none — baseline | — |
| S1 | one point at $(5, 40)$ | central $x$, extreme $y$ → **vertical outlier** |
| S2 | one point at $(30, 47)$ | extreme $x$, *on* the true line → **good leverage** |
| S3 | one point at $(30, 5)$ | extreme $x$, off the line → **bad leverage** |
| S4 | three points near $(29, 5)$ | a cluster of bad-leverage points → **masking** |

Each contaminated sample is fitted by ordinary least squares and, for comparison, by Huber
M-estimation, which down-weights large residuals.

In [ ]:
rng = np.random.default_rng(SEED)
n0, a0, b0 = 60, 2.0, 1.5
x0 = rng.uniform(0, 10, n0)
y0 = a0 + b0*x0 + rng.normal(0, 1.0, n0)          # the clean sample, held fixed throughout

# Each scenario appends contaminating point(s) to the same clean sample
scen = {
    "S0 Clean baseline":                    (x0.copy(), y0.copy()),
    "S1 Vertical outlier (x=5, y=40)":      (np.append(x0, 5.0),  np.append(y0, 40.0)),
    "S2 Good leverage (x=30, on the line)": (np.append(x0, 30.0), np.append(y0, a0 + b0*30.0)),
    "S3 Bad leverage (x=30, y=5)":          (np.append(x0, 30.0), np.append(y0, 5.0)),
    "S4 Masking cluster (3 points)":        (np.concatenate([x0, [28., 29., 30.]]),
                                             np.concatenate([y0, [5., 4., 6.]])),
}

rows = []
for k, (xx, yy) in scen.items():
    Xd = sm.add_constant(xx)
    f = sm.OLS(yy, Xd).fit()
    inf = OLSInfluence(f)                                         # leverage and Cook's distance
    rlm = sm.RLM(yy, Xd, M=sm.robust.norms.HuberT()).fit()        # robust fit for comparison
    # leverage_cutoff = 2(k+1)/n, the usual flag for a high-leverage point (k = 1 slope)
    rows.append(dict(scenario=k, n=len(yy), intercept=f.params[0], slope=f.params[1],
                     se_slope=f.bse[1], t_slope=f.tvalues[1], R2=f.rsquared,
                     resid_sd=np.sqrt(f.mse_resid), max_leverage=inf.hat_matrix_diag.max(),
                     leverage_cutoff=4/len(yy), max_cooks_D=inf.cooks_distance[0].max(),
                     huber_slope=rlm.params[1]))
res2 = pd.DataFrame(rows)
res2["slope_change_vs_clean_%"] = 100*(res2.slope - res2.slope.iloc[0])/res2.slope.iloc[0]
print(f"True data-generating values: intercept = {a0}, slope = {b0}\n")
print(res2.to_string(index=False))

In [ ]:
# The same single point, moved further and further away in y, at two different x positions.
brk = []
for yout in [0, 10, 20, 40, 80, 160, 320]:
    brk.append(dict(y_of_contaminating_point=yout,
                    slope_when_x_is_5 =sm.OLS(np.append(y0, yout), sm.add_constant(np.append(x0, 5.0))).fit().params[1],
                    slope_when_x_is_30=sm.OLS(np.append(y0, yout), sm.add_constant(np.append(x0, 30.0))).fit().params[1]))
brk = pd.DataFrame(brk)
print(f"True slope = {b0}\n")
print(brk.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13.5, 7.4)); axf = ax.ravel()

# One panel per scenario: clean points, contaminating points, true / OLS / Huber lines
grid = np.linspace(0, 32, 50)
for i, (k, (xx, yy)) in enumerate(scen.items()):
    a_ = axf[i]
    f = sm.OLS(yy, sm.add_constant(xx)).fit()
    r = sm.RLM(yy, sm.add_constant(xx), M=sm.robust.norms.HuberT()).fit()
    a_.scatter(x0, y0, s=14, color=BLUE, alpha=.65, label="Clean observations (n = 60)")
    extra = np.arange(n0, len(xx))
    if len(extra):
        a_.scatter(xx[extra], yy[extra], s=65, marker="D", color=RED, edgecolor="k", lw=.6,
                   zorder=5, label="Contaminating point(s)")
    a_.plot(grid, a0 + b0*grid, "k-", lw=1.3, label=f"True line (slope {b0})")
    a_.plot(grid, f.params[0] + f.params[1]*grid, "--", color=RED, lw=1.7,
            label=f"Least squares (slope {f.params[1]:.2f})")
    a_.plot(grid, r.params[0] + r.params[1]*grid, ":", color=GREEN, lw=1.9,
            label=f"Huber M-estimator (slope {r.params[1]:.2f})")
    a_.set(title=k, xlabel="x", ylabel="y", xlim=(-1, 32))
    a_.legend(fontsize=6.4, frameon=False, loc="upper left")

# Last panel: slope as the single contaminating point moves up, at low vs high leverage
axf[5].plot(brk.y_of_contaminating_point, brk.slope_when_x_is_5,  "o-", color=BLUE,
            label="Contaminating point at x = 5 (central)")
axf[5].plot(brk.y_of_contaminating_point, brk.slope_when_x_is_30, "s-", color=RED,
            label="Contaminating point at x = 30 (high leverage)")
axf[5].axhline(b0, color="k", lw=1.2, label=f"True slope = {b0}")
axf[5].set(xlabel="y-value of the single contaminating point", ylabel="Least-squares slope estimate",
           title="Leverage, not outlyingness, causes the damage")
axf[5].legend(fontsize=7, frameon=False)
save_fig("fig_2b_outliers")

#### Baseline replication — three bad-leverage outliers (Answer.pdf)

The group's first pass at 2b drew 100 clean points from $Y_i = 2 + 3X_i + e_i$ with
$X_i, e_i \sim N(0,1)$, then added three points at $(8,-20), (9,-25), (10,-30)$: far out in $x$ and
far below the line. The cell reproduces the written answer exactly (legacy seed 610).

In [ ]:
np.random.seed(610)          # legacy global seed, kept so the numbers match Answer.pdf exactly
xb = np.random.normal(0, 1, 100)
yb = 2 + 3*xb + np.random.normal(0, 1, 100)                     # clean sample: true line y = 2 + 3x
xo, yo = np.array([8., 9., 10.]), np.array([-20., -25., -30.])  # far out in x AND far below the line
clean = sm.OLS(yb, sm.add_constant(xb)).fit()
cont  = sm.OLS(np.concatenate([yb, yo]), sm.add_constant(np.concatenate([xb, xo]))).fit()
# Leverage of the added points compared with the average leverage (k+1)/n
print(pd.DataFrame({"intercept": [clean.params[0], cont.params[0]], "slope": [clean.params[1], cont.params[1]]},
                   index=["without outliers (n = 100)", "with 3 outliers (n = 103)"]).to_string())
print(f"\nLeverage of the three added points: {np.round(OLSInfluence(cont).hat_matrix_diag[-3:], 3)}"
      f"  (average leverage = {2/103:.3f})")

**Results.** A vertical outlier at a central $x$ (S1) barely moves the slope (−2.6%), but it quadruples the residual standard deviation and cuts $R^2$ from 0.95 to 0.51. A high-leverage point that lies on the true line (S2) is harmless and even sharpens the slope estimate (standard error 0.031 against 0.047). A single high-leverage point off the line (S3) cuts the slope by 64%. A cluster of three (S4) cuts it by 89% and leaves it insignificant ($t = 1.81$), yet no single point looks as extreme as in S3 (largest Cook's distance 1.2 against 37), because the three points mask one another. The Huber estimator keeps the slope between 1.47 and 1.57 in S1–S3 but only partly resists the masking cluster (1.22). The sweep table shows the mechanism: moving one point in $y$ at $x = 5$ hardly changes the slope, while at $x = 30$ the slope follows the point. In the baseline replication, three points with leverage 0.18–0.28 (against an average of 0.019) turn a slope of 2.99 into −1.10.

### Question 2c — Numerical confirmation of the high-leverage limit

The report proves algebraically that when $n-1$ points lie on $y=0$ and a single point
$(x_n, y_n)$ is pushed out to $x_n \to \infty$, the least-squares slope satisfies
$\hat\beta \to y_n/x_n$ and the leverage of that point tends to one. The cell below checks both
claims numerically.

In [ ]:
# Question 2c set-up: n-1 = 49 points on the line y = 0 and one point (x_n, y_n = 7)
rng = np.random.default_rng(SEED)
nn = 50
xc, yc, yn = rng.uniform(0, 10, nn-1), np.zeros(nn-1), 7.0

# Push x_n outwards and compare the OLS slope with the ratio y_n / x_n
rows = []
for xn in [10, 1e2, 1e3, 1e4, 1e6, 1e8]:
    f = sm.OLS(np.append(yc, yn), sm.add_constant(np.append(xc, xn))).fit()
    rows.append(dict(x_n=xn, slope_hat=f.params[1], y_n_over_x_n=yn/xn,
                     ratio=f.params[1]/(yn/xn), intercept_hat=f.params[0],
                     leverage_of_outlier=OLSInfluence(f).hat_matrix_diag[-1]))   # h_nn -> 1
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:16.10g}"))
print("\nThe ratio slope_hat / (y_n / x_n) converges to 1 and the leverage converges to 1:")
print("the fitted line is pinned to the single far-out point and the other 49 observations are ignored.")

**Results.** As $x_n$ grows from $10^2$ to $10^8$, the ratio $\hat\beta / (y_n/x_n)$ falls from 1.013 to 1.00000006 and the leverage of the far point rises to 1. The fitted line passes through $(x_n, y_n)$, and its slope is set by $y_n/x_n$ alone, whatever the other 49 observations are.

### Question 3a — Model selection on `FE-GWP1_model_selection_1.csv`

Group 17817 is an odd group number, so the assigned file is `FE-GWP1_model_selection_1.csv`.
Eight selection routes are run: exhaustive best-subset search scored by adjusted
$R^2$, by AIC and by BIC; forward stepwise on AIC and on adjusted $R^2$; backward elimination on BIC; backward elimination
on $t$-statistics; and ten-fold cross-validated prediction error as an out-of-sample referee.

In [ ]:
# Dataset 1 (odd group number): Y and five candidate predictors X1..X5
df = pd.read_csv(f"{DATA}/FE-GWP1_model_selection_1.csv")
pred, y = ["X1", "X2", "X3", "X4", "X5"], df["Y"].values
print("Observations:", df.shape[0], " Variables:", df.shape[1], " Missing values:", int(df.isna().sum().sum()))
print("\nDescriptive statistics"); print(df.describe().T)

# Low correlations among the X's already suggest little collinearity
print("\nPearson correlation matrix"); print(df.corr())

In [ ]:
# --- the unrestricted model, and a multicollinearity check -------------------------------------
full = sm.OLS(y, sm.add_constant(df[pred])).fit()
print(full.summary())

# VIF_j = 1 / (1 - R_j^2), where R_j^2 comes from regressing X_j on the other predictors
vif = pd.DataFrame({"variable": pred,
                    "VIF": [variance_inflation_factor(sm.add_constant(df[pred]).values, i+1) for i in range(5)]})
print("\nVariance inflation factors (values near 1 indicate essentially orthogonal predictors)")
print(vif.to_string(index=False))

In [ ]:
# --- Route 1: exhaustive search over all 2^5 = 32 candidate models ------------------------------
recs = []

# Fit every subset of the five predictors (sizes 0 to 5) and record each criterion
for k in range(6):
    for combo in itertools.combinations(pred, k):
        Xd = sm.add_constant(df[list(combo)]) if combo else np.ones((len(df), 1))
        f = sm.OLS(y, Xd).fit()
        recs.append(dict(size=k, model=" + ".join(combo) if combo else "intercept only",
                         R2=f.rsquared, adj_R2=f.rsquared_adj, AIC=f.aic, BIC=f.bic,
                         resid_sd=np.sqrt(f.mse_resid),
                         # Mallows' Cp = SSR_p / s^2(full) - n + 2p; a good model has Cp close to p
                         Mallows_Cp=f.ssr/full.mse_resid - len(y) + 2*(k+1)))
subs = pd.DataFrame(recs)
print("Best model at every size (scored by AIC):")
print(subs.loc[subs.groupby("size").AIC.idxmin()].to_string(index=False))
print("\nTop six models overall, by AIC:"); print(subs.nsmallest(6, "AIC").to_string(index=False))
print("\nTop six models overall, by BIC:"); print(subs.nsmallest(6, "BIC").to_string(index=False))
print("\nTop six models overall, by adjusted R-squared:"); print(subs.nlargest(6, "adj_R2").to_string(index=False))

In [ ]:
# Akaike weights and BIC posterior model probabilities quantify how decisive the winner is.

# weight = exp(-delta/2) / sum(exp(-delta/2)): the relative likelihood of each model
d = subs.AIC - subs.AIC.min();  subs["delta_AIC"] = d;  subs["Akaike_weight"] = np.exp(-d/2)/np.exp(-d/2).sum()
b = subs.BIC - subs.BIC.min();  subs["delta_BIC"] = b;  subs["BIC_posterior"]  = np.exp(-b/2)/np.exp(-b/2).sum()
print("Akaike weights, top five models");   print(subs.nsmallest(5, "AIC")[["model","AIC","delta_AIC","Akaike_weight"]].to_string(index=False))
print("\nBIC posterior model probabilities, top five models"); print(subs.nsmallest(5, "BIC")[["model","BIC","delta_BIC","BIC_posterior"]].to_string(index=False))

In [ ]:
# --- Routes 2 to 5: stepwise procedures, printing every step so the path is auditable -----------

# Greedy search: at each step try every single addition (forward) or deletion (backward),
# accept the best move only if it improves the criterion, and stop when none does.
def stepwise(crit="aic", direction="forward"):
    # adjusted R^2 is maximised, so it is negated internally and every criterion is minimised
    sgn = -1.0 if crit == "adjr2" else 1.0
    def score(cols):
        Xd = sm.add_constant(df[list(cols)]) if cols else np.ones((len(df), 1))
        f = sm.OLS(y, Xd).fit()
        return f.aic if crit == "aic" else (f.bic if crit == "bic" else -f.rsquared_adj)
    cur = [] if direction == "forward" else list(pred)
    print(f"  start: [{' + '.join(cur) if cur else 'intercept only'}]   {crit.upper()} = {sgn*score(cur):.4f}")
    while True:
        base, cands = score(cur), []
        pool = [p for p in pred if p not in cur] if direction == "forward" else list(cur)
        for p in pool:
            trial = cur + [p] if direction == "forward" else [q for q in cur if q != p]
            cands.append((score(trial), p, trial))
        if not cands: break
        cands.sort()
        for sc, p, _ in cands:
            print(f"    {'add ' if direction=='forward' else 'drop'} {p}:  {crit.upper()} = {sgn*sc:9.4f}"
                  f"   improvement = {base-sc:+7.4f}")
        bs, bp, bt = cands[0]
        if bs < base - 1e-10:
            cur = bt
            print(f"  -> {'ADD' if direction=='forward' else 'DROP'} {bp};  model = [{' + '.join(cur)}]"
                  f"   {crit.upper()} = {sgn*bs:.4f}\n")
        else:
            print(f"  -> no candidate improves the criterion; stop at [{' + '.join(cur)}]\n"); break
    return cur

print("Route 2 — forward stepwise minimising AIC");        fwd_aic = stepwise("aic", "forward")
print("Route 3 — backward elimination minimising BIC");    bwd_bic = stepwise("bic", "backward")
print("Route 4 — forward stepwise maximising adjusted R-squared"); fwd_adj = stepwise("adjr2", "forward")

# Route 5 drops the least significant predictor until every p-value is below 0.05
print("Route 5 — backward elimination on t-statistics at the 5 per cent level")
cur = list(pred)
while True:
    f = sm.OLS(y, sm.add_constant(df[cur])).fit(); pv = f.pvalues[1:]
    print(f"   model [{' + '.join(cur)}]: largest p-value = {pv.max():.4f} on {pv.idxmax()}")
    if pv.max() > 0.05: cur.remove(pv.idxmax())
    else: print(f"   -> every remaining predictor is significant: [{' + '.join(cur)}]\n"); break
bwd_p = cur

In [ ]:
# --- Route 6: ten-fold cross-validated prediction error -----------------------------------------

# Assign each observation to one of 10 folds at random; each fold is held out once
folds = np.random.default_rng(SEED).permutation(np.arange(len(df)) % 10)
cvr = []
for k in range(6):
    for combo in itertools.combinations(pred, k):
        errs = []
        for fo in range(10):
            tr, te = folds != fo, folds == fo
            Xtr = sm.add_constant(df.loc[tr, list(combo)]) if combo else np.ones((tr.sum(), 1))
            Xte = sm.add_constant(df.loc[te, list(combo)]) if combo else np.ones((te.sum(), 1))
            errs.append(y[te] - np.asarray(sm.OLS(y[tr], Xtr).fit().predict(Xte)))
        e = np.concatenate(errs)
        cvr.append(dict(model=" + ".join(combo) if combo else "intercept only", size=k,
                        cv_RMSE=np.sqrt((e**2).mean()), cv_MAE=np.abs(e).mean()))
cvd = pd.DataFrame(cvr)
print("Eight subsets with the lowest out-of-sample error"); print(cvd.nsmallest(8, "cv_RMSE").to_string(index=False))

# Summary: the model chosen by each route
print("\n" + "="*72)
print("AGREEMENT ACROSS ALL SIX SELECTION ROUTES")
print("="*72)
print(pd.DataFrame([
    ("Exhaustive best subset — adjusted R-squared", subs.loc[subs.adj_R2.idxmax(), "model"]),
    ("Exhaustive best subset — AIC",                subs.loc[subs.AIC.idxmin(), "model"]),
    ("Exhaustive best subset — BIC",                subs.loc[subs.BIC.idxmin(), "model"]),
    ("Forward stepwise — AIC",                      " + ".join(sorted(fwd_aic, key=pred.index))),
    ("Backward elimination — BIC",                  " + ".join(sorted(bwd_bic, key=pred.index))),
    ("Forward stepwise — adjusted R-squared",       " + ".join(sorted(fwd_adj, key=pred.index))),
    ("Backward elimination — t-tests at 5%",        " + ".join(sorted(bwd_p,  key=pred.index))),
    ("Ten-fold cross-validation — lowest RMSE",     cvd.loc[cvd.cv_RMSE.idxmin(), "model"]),
], columns=["selection route", "model chosen"]).to_string(index=False))

In [ ]:
# --- the retained model, its robust standard errors, and its diagnostics -------------------------

# The model selected by every route
FINAL = ["X2", "X3", "X4", "X5"]
fin = sm.OLS(y, sm.add_constant(df[FINAL])).fit()
print(fin.summary())

# HC3 standard errors check that inference does not rely on constant error variance
print("\nHeteroskedasticity-consistent (HC3) standard errors")
print(fin.get_robustcov_results(cov_type="HC3").summary().tables[1])

In [ ]:
# Specification checks on the residuals of the retained model
bp = het_breuschpagan(fin.resid, fin.model.exog)
jb = stats.jarque_bera(fin.resid); swt = stats.shapiro(fin.resid)
rs = linear_reset(fin, power=3, test_type="fitted", use_f=True)
inf = OLSInfluence(fin)
print("Diagnostic tests on the retained model")
print(f"   Breusch–Pagan   LM = {bp[0]:.4f}  p = {bp[1]:.4f}   (constant error variance)")
print(f"   Jarque–Bera     JB = {jb.statistic:.4f}  p = {jb.pvalue:.4f}   (normal residuals)")
print(f"   Shapiro–Wilk     W = {swt.statistic:.4f}  p = {swt.pvalue:.4f}")
print(f"   Durbin–Watson   DW = {sm.stats.durbin_watson(fin.resid):.4f}")
print(f"   Ramsey RESET     F = {rs.statistic:.4f}  p = {rs.pvalue:.4f}   (no neglected non-linearity)")
print(f"   Largest Cook's distance = {inf.cooks_distance[0].max():.4f} against the 4/n guide of {4/len(df):.4f}")
print(f"   Largest leverage        = {inf.hat_matrix_diag.max():.4f} against the 2(k+1)/n guide of {2*(len(FINAL)+1)/len(df):.4f}")

# F-test of H0: coefficient on X1 = 0 in the full model, i.e. dropping X1 is justified
R = np.zeros((1, 6)); R[0, pred.index("X1")+1] = 1
ft = full.f_test(R)
print(f"\n   Exclusion test for the discarded predictor X1:  F(1, {int(ft.df_denom)}) = {float(ft.statistic):.4f},"
      f"  p = {float(ft.pvalue):.4f}")

In [ ]:
# Figure 1: how each criterion trades fit against model size, plus the out-of-sample check
fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.8))
bs_ = subs.loc[subs.groupby("size").AIC.idxmin()].sort_values("size")
ax[0].plot(bs_["size"], bs_.AIC, "o-", color=BLUE, label="AIC")
ax[0].plot(bs_["size"], bs_.BIC, "s-", color=RED,  label="BIC")
ax[0].axvline(len(FINAL), color=GREY, ls="--", lw=1.2, label=f"Retained model size = {len(FINAL)}")
ax[0].set(xlabel="Number of predictors in the model", ylabel="Information criterion (lower is better)",
          title="(i) Best model at each size", xticks=range(6)); ax[0].legend(fontsize=7.5, frameon=False)

ax[1].plot(bs_["size"], bs_.adj_R2, "o-", color=GREEN, label="Adjusted R²")
ax[1].plot(bs_["size"], subs.loc[subs.groupby("size").R2.idxmax()].sort_values("size").R2, "^--", color=GREY,
           label="R² (never falls)")
ax[1].axvline(len(FINAL), color=GREY, ls="--", lw=1.2)
ax[1].set(xlabel="Number of predictors in the model", ylabel="Goodness of fit",
          title="(ii) R² versus adjusted R²", xticks=range(6)); ax[1].legend(fontsize=7.5, frameon=False)

# Highlight the retained model among the eight best cross-validated subsets
top = cvd.nsmallest(8, "cv_RMSE").iloc[::-1]
ax[2].barh(range(len(top)), top.cv_RMSE, color=[RED if m == " + ".join(FINAL) else BLUE for m in top.model])
ax[2].set_yticks(range(len(top))); ax[2].set_yticklabels(top.model, fontsize=6.5)
ax[2].set(xlabel="Ten-fold cross-validated RMSE", title="(iii) Out-of-sample error, best eight subsets",
          xlim=(top.cv_RMSE.min()*0.97, top.cv_RMSE.max()*1.01))
save_fig("fig_3a_model_selection")

# Figure 2: residual diagnostics for the retained model
fig, ax = plt.subplots(1, 4, figsize=(14.5, 3.4))
ax[0].scatter(fin.fittedvalues, fin.resid, s=15, color=BLUE); ax[0].axhline(0, color="k", lw=1)
ax[0].set(xlabel="Fitted value of Y", ylabel="Residual", title="Residuals against fitted values")
sm.qqplot(fin.resid, line="45", fit=True, ax=ax[1], markersize=3.5, markerfacecolor=BLUE, markeredgecolor=BLUE)
ax[1].set(title="Normal quantile–quantile plot", xlabel="Theoretical quantile", ylabel="Sample quantile")
ax[2].scatter(fin.fittedvalues, np.sqrt(np.abs(inf.resid_studentized_internal)), s=15, color=GREEN)
ax[2].set(xlabel="Fitted value of Y", ylabel="Root absolute standardised residual", title="Scale–location plot")
ax[3].stem(np.arange(len(df)), inf.cooks_distance[0], markerfmt=".", basefmt=" ", linefmt=BLUE)
ax[3].axhline(4/len(df), color=RED, ls="--", lw=1.2, label="4/n guide")
ax[3].set(xlabel="Observation index", ylabel="Cook's distance", title="Influence diagnostics")
ax[3].legend(fontsize=7.5, frameon=False)
save_fig("fig_3a_diagnostics")

**Results.** All eight routes select the same model, **Y on X2, X3, X4 and X5**: exhaustive search under adjusted $R^2$, AIC and BIC; forward AIC; backward BIC; forward adjusted $R^2$; backward $t$-tests; and ten-fold cross-validation. X1 adds nothing ($t = -0.15$, $p = 0.88$), and dropping it lowers AIC by 2.0 and BIC by 4.6. The predictors are almost orthogonal (every VIF is below 1.06), so the choice is not an artefact of collinearity. The retained model explains 64.9% of the variation in Y (adjusted $R^2 = 0.634$) and passes every diagnostic: Breusch–Pagan $p = 0.65$, Jarque–Bera $p = 0.14$, Durbin–Watson 1.97, RESET $p = 0.94$. X5 is the weakest link. It is significant with conventional standard errors ($p = 0.021$) but only at the 10% level with HC3 robust errors ($p = 0.078$), and BIC still gives the three-predictor model X2 + X3 + X4 a posterior probability of 0.34, against 0.57 for the retained model.

### Question 3b — Why BIC and not AIC: the over-fitting probability

Comparing the true model against the same model plus one redundant predictor, the likelihood-ratio
statistic $LR$ is asymptotically $\chi^2_1$ under the null that the extra coefficient is zero. A
criterion prefers the larger model precisely when $LR$ exceeds its penalty gap: $2$ for AIC and
$\ln n$ for BIC. Hence

$$P(\text{AIC over-fits}) \to P(\chi^2_1 > 2) = 0.1573 \quad\text{for every } n, \qquad
  P(\text{BIC over-fits}) = P(\chi^2_1 > \ln n) \to 0 .$$

In [ ]:
# Theory: the extra parameter is kept when LR > 2 (AIC) or LR > ln n (BIC), with LR ~ chi2(1)
print(f"AIC penalty gap = 2        ->  P(over-fit) = P(chi2_1 > 2) = {stats.chi2.sf(2,1):.4f}  for every n")
for n in [50, 100, 500, 1000, 10000, 100000, 1000000]:
    print(f"BIC penalty gap = ln(n) = {np.log(n):7.3f}  (n = {n:>7,})  ->  P(over-fit) = {stats.chi2.sf(np.log(n),1):.5f}")

# Monte Carlo: z is irrelevant by construction, so choosing the larger model is an error
rows = []
for n in [50, 100, 250, 500, 1000, 5000, 20000]:
    rg = np.random.default_rng(SEED + n); ka = kb = kr = 0
    for _ in range(2000):
        x_, z_ = rg.normal(size=n), rg.normal(size=n)
        y_ = 1 + 2*x_ + rg.normal(size=n)                      # z_ is irrelevant by construction
        m0 = sm.OLS(y_, sm.add_constant(x_)).fit()
        m1 = sm.OLS(y_, sm.add_constant(np.column_stack([x_, z_]))).fit()
        # count how often each criterion prefers the over-parameterised model
        ka += m1.aic < m0.aic; kb += m1.bic < m0.bic; kr += m1.rsquared_adj > m0.rsquared_adj
    rows.append(dict(n=n, AIC_overfit_rate=ka/2000, BIC_overfit_rate=kb/2000,
                     adjR2_overfit_rate=kr/2000, BIC_theoretical=stats.chi2.sf(np.log(n), 1)))
mcsel = pd.DataFrame(rows)
print("\nMonte Carlo, 2,000 replications per sample size"); print(mcsel.to_string(index=False))

In [ ]:
# (i) simulated over-fitting rates against theory; (ii) where each penalty cuts the chi2(1) density
fig, ax = plt.subplots(1, 2, figsize=(11, 3.7))
ax[0].plot(mcsel.n, mcsel.AIC_overfit_rate, "o-", color=RED, label="AIC (simulated)")
ax[0].axhline(stats.chi2.sf(2, 1), color=RED, ls=":", lw=1.3, label="AIC theory = 0.157")
ax[0].plot(mcsel.n, mcsel.BIC_overfit_rate, "s-", color=BLUE, label="BIC (simulated)")
ax[0].plot(mcsel.n, mcsel.BIC_theoretical, "--", color=BLUE, lw=1.2, label="BIC theory = P(χ²₁ > ln n)")
ax[0].plot(mcsel.n, mcsel.adjR2_overfit_rate, "^-", color=PURPLE, label="Adjusted R² (simulated)")
ax[0].set(xscale="log", xlabel="Sample size n (logarithmic scale)",
          ylabel="Probability of choosing the over-parameterised model",
          title="(i) Only the BIC error rate goes to zero"); ax[0].legend(fontsize=7, frameon=False)

xg = np.linspace(0, 12, 300)
ax[1].plot(xg, stats.chi2.pdf(xg, 1), color="k", lw=1.3, label="χ²₁ density of the LR statistic")
ax[1].fill_between(xg, 0, stats.chi2.pdf(xg, 1), where=xg > 2, color=RED, alpha=.35,
                   label="AIC over-fits (LR > 2): 15.7%")
ax[1].fill_between(xg, 0, stats.chi2.pdf(xg, 1), where=xg > np.log(1000), color=BLUE, alpha=.55,
                   label="BIC over-fits at n = 1000 (LR > 6.91): 0.9%")
ax[1].set(ylim=(0, .6), xlabel="Likelihood-ratio statistic for the redundant predictor",
          ylabel="Density", title="(ii) Where the two penalties cut")
ax[1].legend(fontsize=7, frameon=False)
save_fig("fig_3b_aic_vs_bic")

**Results.** For one extra parameter, the penalty is fixed at 2 under AIC but grows as $\ln n$ under BIC. The simulation matches the theory. AIC keeps the redundant regressor in about 16% of samples whatever the sample size (0.14–0.18), whereas BIC's error rate falls from 5.4% at $n = 50$ to 0.05% at $n = 20{,}000$. Adjusted $R^2$ does worse than both (about 32%, the probability that $|t| > 1$). **BIC is consistent: it picks the true model with probability tending to one. AIC is not.**

### Problem 4 — The log-wage equation (Questions 4c–4h)

$$\ln(\text{wage}) = \beta_0 + \beta_1 \text{educ} + \beta_2 \text{gender} + \beta_3 \text{age}
   + \beta_4 \text{age}^2 + \beta_5 \text{degree} + \varepsilon, \qquad n = 2{,}000,\; R^2 = 0.44 .$$

The cell below reconstructs the inference implied by the published coefficient table, converts
log-point coefficients into exact percentage effects through $100(e^{\hat\beta}-1)$, and locates the
turning point of the age profile.

In [ ]:
# Coefficient table as published in the brief (n = 2,000, five regressors)
wage = pd.DataFrame({
    "variable": ["intercept", "educ", "gender (female = 1)", "age", "age squared", "degree"],
    "coefficient": [0.85, 0.08, -0.15, 0.19, -0.002, 0.22],
    "std_error":   [0.12, 0.01,  0.10, 0.06,  0.0003, 0.04],
    "t_stat":      [7.08, 8.00, -1.50, 3.17, -6.67,  5.50]})
n_w, k_w, R2 = 2000, 5, 0.44

# Two-sided 5% critical value with n - k - 1 degrees of freedom
tc = stats.t.ppf(0.975, n_w - k_w - 1)
wage["p_value"]     = 2*stats.t.sf(np.abs(wage.t_stat), n_w-k_w-1)
wage["CI_low"]      = wage.coefficient - tc*wage.std_error
wage["CI_high"]     = wage.coefficient + tc*wage.std_error

# In a log-level model a coefficient b means a 100(e^b - 1)% change in the wage
wage["exact_pct"]   = 100*(np.exp(wage.coefficient)-1)
wage["pct_CI_low"]  = 100*(np.exp(wage.CI_low)-1)
wage["pct_CI_high"] = 100*(np.exp(wage.CI_high)-1)
print(f"Critical value of t at the 5 per cent level with {n_w-k_w-1} degrees of freedom: {tc:.4f}\n")
print(wage.to_string(index=False))

# Overall F-statistic recovered from R^2: F = (R^2 / k) / ((1 - R^2) / (n - k - 1))
F = (R2/k_w)/((1-R2)/(n_w-k_w-1))
print(f"\nOverall F({k_w}, {n_w-k_w-1}) = {F:.2f},  p = {stats.f.sf(F, k_w, n_w-k_w-1):.3e}")
print(f"Adjusted R-squared = {1-(1-R2)*(n_w-1)/(n_w-k_w-1):.4f}")

In [ ]:
# Question 4f: d ln(wage) / d age = b3 + 2 b4 age, which is zero at the peak
b3, b4 = 0.19, -0.002
age_star = -b3/(2*b4)
print("Question 4f — turning point of the age–earnings profile")
print(f"   age* = -b3 / (2 b4) = -{b3} / (2 × {b4}) = {age_star:.2f} years")
print(f"   second derivative = 2 b4 = {2*b4} < 0, so the profile is concave and {age_star:.1f} is a maximum")

# Delta method for age* = -b3 / (2 b4). corr(b3, b4) is not published, so -0.95 is an
# assumption: age and age^2 are nearly collinear, so their estimates move in opposite directions
se3, se4, rho34 = 0.06, 0.0003, -0.95
se_star = age_star*np.sqrt((se3/b3)**2 + (se4/b4)**2 - 2*rho34*se3*se4/(b3*b4))
print(f"   delta-method standard error assuming corr(b3, b4) = {rho34}:  {se_star:.2f} years"
      f"  ->  95% interval [{age_star-1.96*se_star:.1f}, {age_star+1.96*se_star:.1f}] years")

print("\nQuestion 4d — return to a year of schooling")
print(f"   exact effect = 100(exp(0.08) - 1) = {100*(np.exp(0.08)-1):.2f}% per year; "
      f"four years = {100*(np.exp(4*0.08)-1):.2f}%")
print("\nQuestion 4e — the gender coefficient")
print(f"   point estimate -0.15  ->  exact gap {100*(np.exp(-0.15)-1):.2f}%;  t = -1.50, p = {wage.p_value[2]:.4f}")
print(f"   95% interval on the log scale [{wage.CI_low[2]:.4f}, {wage.CI_high[2]:.4f}]"
      f"  ->  in percentage terms [{wage.pct_CI_low[2]:.1f}%, {wage.pct_CI_high[2]:+.1f}%]")
print(f"   smallest gap detectable at 5% given se = 0.10: {tc*0.10:.4f} log points "
      f"({100*(np.exp(tc*0.10)-1):.1f}%)")
print(f"   Kennedy (1981) bias-corrected estimate: {100*(np.exp(-0.15 - 0.10**2/2)-1):.2f}%")
print("\nQuestion 4g — the advanced-degree premium")
print(f"   exact premium = 100(exp(0.22) - 1) = {100*(np.exp(0.22)-1):.2f}%;"
      f"  95% interval [{wage.pct_CI_low[5]:.1f}%, {wage.pct_CI_high[5]:.1f}%]")
print(f"   Kennedy (1981) bias-corrected estimate: {100*(np.exp(0.22 - 0.04**2/2)-1):.2f}%")

# The return to one more year of age falls linearly and turns negative after the peak
print("\nMarginal effect of age,  d ln(wage)/d age = 0.19 - 0.004 × age")
print(pd.DataFrame({"age": [25, 30, 35, 40, 45, 47.5, 50, 55, 60, 65]})
        .assign(marginal_effect=lambda t: b3 + 2*b4*t.age,
                pct_per_extra_year=lambda t: 100*(np.exp(t.marginal_effect)-1)).to_string(index=False))

In [ ]:
# (i) age profile and its peak; (ii) marginal effect of age; (iii) key effects with 95% intervals
fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.8))
ag = np.linspace(18, 70, 300)
ax[0].plot(ag, b3*ag + b4*ag**2, color=BLUE, lw=1.9)
ax[0].axvline(age_star, color=RED, ls="--", lw=1.3, label=f"Peak at {age_star:.1f} years")
ax[0].scatter([age_star], [b3*age_star + b4*age_star**2], color=RED, s=40, zorder=5)
ax[0].set(xlabel="Age (years)", ylabel="Contribution of age to ln(wage)",
          title="(i) Concave age–earnings profile"); ax[0].legend(fontsize=7.5, frameon=False)

ax[1].plot(ag, 100*(np.exp(b3 + 2*b4*ag)-1), color=GREEN, lw=1.9)
ax[1].axhline(0, color="k", lw=1); ax[1].axvline(age_star, color=RED, ls="--", lw=1.2)
ax[1].set(xlabel="Age (years)", ylabel="Wage growth per additional year of age (%)",
          title="(ii) Marginal effect of age")

# Coefficient plot rows: schooling, gender and degree (rows 1, 2 and 5 of the table)
names = ["One more year of schooling", "Female relative to male", "Master's degree or higher"]
idx = [1, 2, 5]
pts = [wage.exact_pct[i] for i in idx]
lo  = [wage.pct_CI_low[i] for i in idx]; hi = [wage.pct_CI_high[i] for i in idx]
ax[2].errorbar(pts, range(3), xerr=[np.array(pts)-np.array(lo), np.array(hi)-np.array(pts)],
               fmt="o", color="k", capsize=3, ms=5)
for i, cc in enumerate([BLUE, GREY, GREEN]): ax[2].scatter([pts[i]], [i], color=cc, s=60, zorder=5)
ax[2].axvline(0, color=RED, ls="--", lw=1.2)
ax[2].set_yticks(range(3)); ax[2].set_yticklabels(names, fontsize=8)
ax[2].set(xlabel="Effect on the hourly wage (%), with a 95% confidence interval",
          title="(iii) Estimated wage effects")
save_fig("fig_4_wage_model")

In [ ]:
# Question 4h — why "male" and "female" cannot both enter alongside an intercept
rg = np.random.default_rng(SEED)

# Simulated log wages with a known female gap of -0.15
fem = rg.integers(0, 2, 500); male = 1 - fem
lw  = 3.0 - 0.15*fem + rg.normal(0, 0.3, 500)

# Constant + male + female: male + female = 1 = the constant column, i.e. perfect collinearity
Xtrap = np.column_stack([np.ones(500), male, fem])
print(f"Design matrix [constant, male, female] has {Xtrap.shape[1]} columns but rank {np.linalg.matrix_rank(Xtrap)};"
      f"  det(X'X) = {np.linalg.det(Xtrap.T @ Xtrap):.3e}  ->  X'X cannot be inverted")

# The two valid parameterisations: drop one dummy, or drop the intercept
m_ok = sm.OLS(lw, sm.add_constant(fem)).fit()
m_nc = sm.OLS(lw, np.column_stack([male, fem])).fit()
print(f"\nIntercept plus one dummy:      constant = {m_ok.params[0]:.4f},  female = {m_ok.params[1]:.4f}")
print(f"Two dummies and no intercept:  male     = {m_nc.params[0]:.4f},  female = {m_nc.params[1]:.4f}")
print(f"The implied gap is identical either way: {m_nc.params[1]-m_nc.params[0]:.4f}")

**Results.** These are the quantities the report uses. One more year of schooling raises the hourly wage by 8.3% (95% CI 6.2–10.5%). The gender coefficient implies women earn 13.9% less, but it is not statistically significant ($t = -1.50$, $p = 0.13$). Its interval runs from −29.3% to +4.7%, so these data can neither establish a gap nor rule out a sizeable one. Wages peak at age 47.5. A Master's degree or higher adds 24.6% (95% CI 15.2–34.8%). With an intercept plus both a male and a female dummy, the design matrix has rank 2 but three columns (the dummy-variable trap). Dropping either the intercept or one dummy gives the same estimated gap.

### Question 5c — Visual inspection, unit roots, Johansen tests and VECM

The underlying processes are $X_t = w_t + \eta_t$, $Y_t = 0.5 w_t + \theta_t$, $Z_t = -w_t + \chi_t$
with $w_t$ a driftless random walk. Theory (Question 5b) therefore implies one common stochastic
trend and a cointegration rank of two, with cointegrating space
$\{(a,b,c): a + 0.5b - c = 0\}$.

In [ ]:
# The file stores each record as a single quoted string, so it is parsed line by line.
lines = [l.strip().strip('"') for l in
         open(f"{DATA}/FE-GWP1_dataset_Prob5.csv", encoding="utf-8-sig").read().splitlines() if l.strip()]
p5 = pd.DataFrame([[float(v) for v in l.split(",")] for l in lines[1:]], columns=lines[0].split(","))
p5["Period"] = p5["Period"].astype(int); p5 = p5.set_index("Period")

# First differences, used to check that each series is I(1)
dif = p5.diff().dropna(); dif.columns = ["dX", "dY", "dZ"]

print("Observations:", len(p5), " Periods:", p5.index.min(), "to", p5.index.max(),
      " Missing values:", int(p5.isna().sum().sum()))
print("\nDescriptive statistics"); print(p5.describe().T)
print("\nCorrelation in levels (inflated by the shared stochastic trend)"); print(p5.corr())
print("\nCorrelation in first differences"); print(dif.corr())
print("\nSub-sample means by quarter of the sample — a level that wanders is the signature of a stochastic trend")

# Quarter-by-quarter means: the level of a random walk drifts instead of reverting
for nm in ["X", "Y", "Z"]:
    print(f"   {nm}: " + ", ".join(f"{np.mean(s):7.3f}" for s in np.array_split(p5[nm].values, 4)))

In [ ]:
# --- 5c step 1: visual inspection ----------------------------------------------------------------
fig, ax = plt.subplots(2, 3, figsize=(14, 6.6))

# Top row: levels, which should wander; bottom row: first differences, which should look like noise
for i, (nm, col) in enumerate(zip(["X", "Y", "Z"], [BLUE, GREEN, PURPLE])):
    ax[0, i].plot(p5.index, p5[nm], color=col, lw=1.0)
    ax[0, i].axhline(p5[nm].mean(), color=RED, ls="--", lw=1, label=f"Sample mean = {p5[nm].mean():.2f}")
    ax[0, i].set(xlabel="Period (t)", ylabel=f"{nm}(t)",
                 title=f"{nm} in levels — wandering, no mean reversion")
    ax[0, i].legend(fontsize=7, frameon=False)
    ax[1, i].plot(dif.index, dif[f"d{nm}"], color=col, lw=0.8)
    ax[1, i].axhline(0, color=RED, ls="--", lw=1)
    ax[1, i].set(xlabel="Period (t)", ylabel=f"Δ{nm}(t)", title=f"Δ{nm} — stable mean and variance")
save_fig("fig_5c_series")

In [ ]:
# --- 5c step 2: unit-root testing, levels then first differences ---------------------------------

# ADF has H0 = unit root and KPSS has H0 = stationarity. Running both guards against low power:
# an I(1) series should fail to reject under ADF and reject under KPSS.
def ur_table(data):
    rows = []
    for nm in data.columns:
        s = data[nm].dropna().values
        for reg, label in [("c", "constant"), ("ct", "constant and trend")]:
            a = adfuller(s, regression=reg, autolag="AIC")
            rows.append(dict(series=nm, test="ADF", specification=label, statistic=a[0], p_value=a[1],
                             lags=a[2], cv_1pct=a[4]["1%"], cv_5pct=a[4]["5%"],
                             conclusion="unit root rejected" if a[1] < .05 else "unit root NOT rejected"))
        for reg, label in [("c", "level stationary"), ("ct", "trend stationary")]:
            k = kpss(s, regression=reg, nlags="auto")
            rows.append(dict(series=nm, test="KPSS", specification=label, statistic=k[0], p_value=k[1],
                             lags=k[2], cv_1pct=k[3]["1%"], cv_5pct=k[3]["5%"],
                             conclusion="stationarity rejected" if k[1] < .05 else "stationarity NOT rejected"))
    return pd.DataFrame(rows)

# Expected: a unit root in levels and none in first differences
print("LEVELS");            print(ur_table(p5[["X", "Y", "Z"]]).to_string(index=False))
print("\nFIRST DIFFERENCES"); print(ur_table(dif).to_string(index=False))

In [ ]:
# --- 5c: the combinations predicted by Question 5b, tested directly -------------------------------

# If the theory holds, these combinations cancel the common trend w(t) and are stationary;
# X + Y does not cancel it and serves as a control
combos = {"X − 2Y     (weights  1, −2,   0)": p5.X - 2*p5.Y,
          "X + Z      (weights  1,  0,   1)": p5.X + p5.Z,
          "Y + 0.5Z   (weights  0,  1, 0.5)": p5.Y + 0.5*p5.Z,
          "X + Y      (outside the space)  ": p5.X + p5.Y}
for nm, s in combos.items():
    a = adfuller(s.values, regression="c", autolag="AIC")
    print(f"   {nm}   ADF = {a[0]:8.4f}   p = {a[1]:.4f}   "
          f"{'STATIONARY' if a[1] < .05 else 'unit root remains'}")

In [ ]:
# --- 5c step 3: lag order for the system ----------------------------------------------------------

# Information criteria for the number of lagged differences in the VECM
so = select_order(p5[["X", "Y", "Z"]].values, maxlags=8, deterministic="ci")
print(so.summary())
print("Selected number of lagged differences:", so.selected_orders)
K_DIFF = 1      # all criteria choose zero; one lag is carried as a conservative guard

In [ ]:
# --- 5c step 4: Johansen trace and maximum-eigenvalue tests ---------------------------------------

# Johansen test with kd lagged differences; det = -1 means no deterministic terms, 0 a constant
def johansen_report(kd, det, show=True):
    jo = coint_johansen(p5[["X", "Y", "Z"]].values, det_order=det, k_ar_diff=kd)
    eig = np.real(jo.eig)
    tr = pd.DataFrame({"null_hypothesis": [f"rank <= {i}" for i in range(3)], "eigenvalue": eig,
                       "trace_statistic": np.real(jo.lr1),
                       "cv_90pct": jo.cvt[:, 0], "cv_95pct": jo.cvt[:, 1], "cv_99pct": jo.cvt[:, 2]})
    tr["reject_5pct"] = tr.trace_statistic > tr.cv_95pct
    tr["reject_1pct"] = tr.trace_statistic > tr.cv_99pct
    me = pd.DataFrame({"null_hypothesis": [f"rank = {i}" for i in range(3)], "eigenvalue": eig,
                       "max_eigen_statistic": np.real(jo.lr2),
                       "cv_90pct": jo.cvm[:, 0], "cv_95pct": jo.cvm[:, 1], "cv_99pct": jo.cvm[:, 2]})
    me["reject_5pct"] = me.max_eigen_statistic > me.cv_95pct
    me["reject_1pct"] = me.max_eigen_statistic > me.cv_99pct
    # Tests run in sequence from r = 0; the rank is the first null that is not rejected
    pick = lambda c: int(np.argmin(c.values)) if (~c).any() else 3
    if show:
        print(f"### lagged differences = {kd}, deterministic order = {det}")
        print("\nTRACE statistic — test sequentially and stop at the first non-rejection")
        print(tr.to_string(index=False))
        print("\nMAXIMUM-EIGENVALUE statistic"); print(me.to_string(index=False))
        print(f"\nRank chosen by trace at 5% = {pick(tr.reject_5pct)}, at 1% = {pick(tr.reject_1pct)};"
              f" by maximum eigenvalue at 5% = {pick(me.reject_5pct)}")
        print(f"Eigenvalues {np.round(eig,5)} — the second-to-third ratio is {eig[1]/eig[2]:.1f}")
    return jo, tr, me, pick(tr.reject_5pct), pick(tr.reject_1pct), pick(me.reject_5pct)

# Headline specification: one lagged difference and a constant
jo, tr1, me1, *_ = johansen_report(K_DIFF, 0)

# Eigenvectors are the candidate cointegrating vectors, identified only up to scale
bmat = pd.DataFrame(np.real(jo.evec), index=["X", "Y", "Z"], columns=["v1", "v2", "v3"])
print("\nUnnormalised eigenvectors:"); print(bmat)
print("Rescaled so that the X element equals one:"); print(bmat/bmat.iloc[0])

In [ ]:
# Is the rank decision sensitive to the lag order or the deterministic specification?
rob = []

# Re-run the rank tests across lag orders and deterministic terms to check the decision is robust
for kd in [0, 1, 2, 4]:
    for det in [-1, 0]:
        _, t_, m_, a5, a1, b5 = johansen_report(kd, det, show=False)
        rob.append(dict(lagged_differences=kd, deterministic_order=det, trace_rank_5pct=a5,
                        trace_rank_1pct=a1, max_eigen_rank_5pct=b5,
                        last_trace_statistic=t_.trace_statistic.iloc[2], its_cv_95pct=t_.cv_95pct.iloc[2]))
print(pd.DataFrame(rob).to_string(index=False))

print("\nSequential rank selection reported by the library")
for meth in ["trace", "maxeig"]:
    print(select_coint_rank(p5[["X","Y","Z"]].values, det_order=0, k_ar_diff=K_DIFF, method=meth).summary())
print(select_coint_rank(p5[["X","Y","Z"]].values, det_order=0, k_ar_diff=K_DIFF,
                        method="trace", signif=0.01).summary())

# Adopted rank; the results note explains why the borderline r <= 2 rejection is not followed
RANK = 2
print(f"\nRANK ADOPTED: r = {RANK}, implying {3-RANK} common stochastic trend.")
print(f"The final trace statistic is {tr1.trace_statistic.iloc[2]:.4f} against a 5% critical value of "
      f"{tr1.cv_95pct.iloc[2]:.4f} — a rejection by only {tr1.trace_statistic.iloc[2]-tr1.cv_95pct.iloc[2]:.4f}, "
      f"and no rejection at all at the 1% level ({tr1.cv_99pct.iloc[2]:.4f}).")

In [ ]:
# --- 5c step 5: VECM estimation --------------------------------------------------------------------

# deterministic='ci' places the constant inside the cointegrating relations
vr = VECM(p5[["X", "Y", "Z"]], k_ar_diff=K_DIFF, coint_rank=RANK, deterministic="ci").fit()
print(vr.summary())

In [ ]:
print("Loading (speed-of-adjustment) matrix alpha, one row per equation")
print(pd.DataFrame(vr.alpha, index=["ΔX", "ΔY", "ΔZ"], columns=["ec1", "ec2"]))
print("\nCointegrating matrix beta, normalised so that the top 2×2 block is the identity")
print(pd.DataFrame(vr.beta, index=["X", "Y", "Z"], columns=["ec1", "ec2"]))

# Test the estimated Z loadings against the values Question 5b predicts
sb = np.asarray(vr.stderr_coint)
z1 = (vr.beta[2, 0] - 1.0)/sb[2, 0]; z2 = (vr.beta[2, 1] - 0.5)/sb[2, 1]
print("\nQuestion 5b predicts, under this normalisation, beta = [[1, 0], [0, 1], [1, 0.5]].")
print(f"   estimated Z loading on ec1 = {vr.beta[2,0]:.4f} (se {sb[2,0]:.4f});  "
      f"H0 = 1.0  ->  z = {z1:+.4f},  p = {2*stats.norm.sf(abs(z1)):.4f}")
print(f"   estimated Z loading on ec2 = {vr.beta[2,1]:.4f} (se {sb[2,1]:.4f});  "
      f"H0 = 0.5  ->  z = {z2:+.4f},  p = {2*stats.norm.sf(abs(z2)):.4f}")
print(f"   joint chi-square(2) = {z1**2+z2**2:.4f},  p = {stats.chi2.sf(z1**2+z2**2, 2):.4f}"
      "  ->  the theoretical cointegrating space is not rejected")

# Long-run matrix Pi = alpha beta'; its rank must equal the cointegration rank
Pi = vr.alpha @ vr.beta.T
print("\nLong-run impact matrix Pi = alpha · beta'")
print(pd.DataFrame(Pi, index=["ΔX", "ΔY", "ΔZ"], columns=["X", "Y", "Z"]))
print(f"Numerical rank of Pi = {np.linalg.matrix_rank(Pi, tol=1e-8)}, as required for r = {RANK}")
print("\nShort-run coefficient matrix Gamma on the lagged differences")
print(pd.DataFrame(vr.gamma, index=["ΔX", "ΔY", "ΔZ"], columns=["L1.ΔX", "L1.ΔY", "L1.ΔZ"]))
print("\nResidual correlation matrix")
print(pd.DataFrame(np.corrcoef(vr.resid.T), index=["ΔX", "ΔY", "ΔZ"], columns=["ΔX", "ΔY", "ΔZ"]))

In [ ]:
# Equilibrium errors beta' y(t): stationary if the rank and vectors are right
ec = p5[["X", "Y", "Z"]].values @ vr.beta
print("Stationarity and persistence of the fitted equilibrium errors")
for i in range(RANK):
    a = adfuller(ec[:, i], regression="c", autolag="AIC")
    ar1 = sm.OLS(ec[1:, i], sm.add_constant(ec[:-1, i])).fit().params[1]
    hl = np.log(0.5)/np.log(abs(ar1)) if 0 < abs(ar1) < 1 else np.nan
    print(f"   ec{i+1}:  ADF = {a[0]:8.4f}  p = {a[1]:.4f}   AR(1) coefficient = {ar1:+.4f}"
          f"   half-life of a shock = {hl:.2f} periods")

# Residual autocorrelation check for each VECM equation
print("\nLjung–Box test at ten lags on the VECM residuals")
for i, nm in enumerate(["ΔX", "ΔY", "ΔZ"]):
    lb = acorr_ljungbox(vr.resid[:, i], lags=[10], return_df=True)
    print(f"   {nm}:  Q(10) = {lb['lb_stat'].iloc[0]:.4f}   p = {lb['lb_pvalue'].iloc[0]:.4f}"
          f"   -> {'no residual autocorrelation' if lb['lb_pvalue'].iloc[0] > .05 else 'autocorrelation remains'}")

# VECM rewritten as a VAR in levels: with r = 2 the companion matrix needs exactly 3 - r = 1 unit root
A = vr.var_rep; kk = A.shape[0]
comp = np.zeros((3*kk, 3*kk))
for i in range(kk): comp[:3, i*3:(i+1)*3] = A[i]
if kk > 1: comp[3:, :3*(kk-1)] = np.eye(3*(kk-1))
mods = np.sort(np.abs(np.linalg.eigvals(comp)))[::-1]
print(f"\nCompanion-matrix eigenvalue moduli: {np.round(mods, 4)}")
print(f"Exactly {int(np.sum(np.round(mods,2)==1.0))} modulus equals one — matching {3-RANK} common "
      f"stochastic trend for r = {RANK}.")

In [ ]:
# (i) levels; (ii) theoretical cointegrating combinations against a non-cointegrating one;
# (iii) estimated equilibrium errors
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for nm, col in zip(["X", "Y", "Z"], [BLUE, GREEN, PURPLE]):
    ax[0].plot(p5.index, p5[nm], color=col, lw=1, label=nm)
ax[0].set(xlabel="Period (t)", ylabel="Level", title="(i) All three series share one common trend")
ax[0].legend(fontsize=7.5, frameon=False)

for nm, s, col in [("X − 2Y", p5.X-2*p5.Y, BLUE), ("X + Z", p5.X+p5.Z, GREEN),
                   ("X + Y (not cointegrating)", p5.X+p5.Y, RED)]:
    ax[1].plot(p5.index, s, lw=.9, color=col, label=nm)
ax[1].axhline(0, color="k", lw=1)
ax[1].set(xlabel="Period (t)", ylabel="Value of the linear combination",
          title="(ii) Two theoretical combinations are mean-reverting")
ax[1].legend(fontsize=7, frameon=False)

for i, col in zip(range(RANK), [BLUE, GREEN]):
    ax[2].plot(p5.index, ec[:, i], lw=.9, color=col, label=f"Equilibrium error {i+1}")
ax[2].axhline(0, color="k", lw=1)
ax[2].set(xlabel="Period (t)", ylabel="Deviation from long-run equilibrium",
          title=f"(iii) Estimated equilibrium errors, rank r = {RANK}")
ax[2].legend(fontsize=7, frameon=False)
save_fig("fig_5c_cointegration")

**Results.** All three series are I(1). In levels the ADF test cannot reject a unit root ($p \ge 0.12$) and KPSS rejects stationarity ($p \le 0.01$); in first differences the ADF test rejects ($p < 0.001$) and KPSS does not. The two combinations that theory implies, X − 2Y and X + Z, are stationary (ADF $p = 0.0007$ and $p < 0.0001$), while X + Y, which lies outside the cointegrating space, is not ($p = 0.28$). The Johansen tests reject $r = 0$ and $r \le 1$ decisively (trace 262.1 and 128.8 against 5% critical values of 29.8 and 15.5). The test of $r \le 2$ is borderline: 3.93 against 3.84 at 5%, not significant at 1% (6.63), and not rejected in 5 of the 8 specifications tried, including the zero-lag specification that every information criterion selects. A rank of 3 would mean the series themselves are stationary, which contradicts the unit-root evidence, so **$r = 2$ is adopted: one common stochastic trend, as theory predicts.** The estimated cointegrating vectors are (1, 0, 1.006) and (0, 1, 0.415), against theoretical values of (1, 0, 1) and (0, 1, 0.5); the joint test does not reject the theory ($p = 0.31$). The equilibrium errors are stationary, the VECM residuals show no autocorrelation (Ljung–Box $p \ge 0.50$), and the companion matrix has exactly one unit root.